# sorted-computational-graph composite — cx22: build sorted graph from a MiniTensor — DFS over recipe.parents, reversed

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `dfs-three-set-toposort`, `sorted-computational-graph`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "sorted-computational-graph"
DD_ATOM_IDS = ["dfs-three-set-toposort", "sorted-computational-graph"]
DD_SUBTOPICS = ["Backprop: DFS three-set toposort", "Backprop: Sorted computation graph"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Composing DFS toposort with the sorted-computational-graph wrapper

Two atoms, one helper:

- **`dfs-three-set-toposort`** — generic DFS that returns descendants
  of a node in deps-first order (root LAST).
- **`sorted-computational-graph`** — wrap that helper for MiniTensors:
  walk via `recipe.parents.values()`, then REVERSE so the end-node
  comes FIRST (the order the reverse pass wants).

```python
def sorted_computational_graph(tensor):
    def get_parents(t_):
        if t_.recipe is None: return []      # leaves: no parents
        return list(t_.recipe.parents.values())
    return dfs_topo_sort(tensor, get_parents)[::-1]
```

**Why the two-layer split.** The DFS topo-sort is generic — reusable
for any kind of DAG (forward graph, computation graph, build-system
dependency graph). The MiniTensor-specific bit (`recipe.parents` is the
parent set; leaves have no recipe) is isolated in the `get_parents`
closure. Same DFS skeleton works for both forward and reverse
traversals.

**The `[::-1]` is the orientation flip.** `dfs_topo_sort` produces
deps-first (root LAST). Reverse-pass wants end-node FIRST so it can
seed `grads = {id(end): end_grad}` and walk outward. Reversing the
deps-first list is the cheapest way to flip orientation without writing
a second sort.

### Composite Exercise — build sorted graph from a MiniTensor — DFS over recipe.parents, reversed

**Atoms exercised together**: `dfs-three-set-toposort`, `sorted-computational-graph`

Implement TWO helpers that compose:

**1. `cx22_topo_sort(root, get_children)`** — generic three-colour DFS topological sort. Returns descendants of `root` such that `root` is LAST (deps-first). Raises `ValueError` on a cycle.

**2. `cx22_sorted_graph(tensor)`** — wrap the above for MiniTensors:
- Define `get_parents(t_)`: return `[]` if `t_.recipe is None`, else `list(t_.recipe.parents.values())`.
- Call `cx22_topo_sort(tensor, get_parents)` and REVERSE the result.

**Contract for `cx22_sorted_graph(tensor)`.**
- `result[0] is tensor` — the end node comes FIRST.
- `result[-1].recipe is None` — last is some leaf.
- Each unique node in the graph appears exactly once.
- Reverse-iteration order: every node appears BEFORE all of its parents (so a parent's accumulator is fully summed by the time the loop reaches it).

**Recipe / MiniTensor are provided in the test cell.**

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx22_topo_sort(root, get_children):
    """Three-colour DFS topo sort, deps-first (root LAST)."""
    raise NotImplementedError()

def cx22_sorted_graph(tensor):
    """Reverse-topological order of MiniTensor graph (end node FIRST)."""
    raise NotImplementedError()

def _test_cx22():
    from dataclasses import dataclass, field
    from typing import Any, Callable, Optional

    @dataclass
    class Recipe:
        func: Optional[Callable] = None
        args: tuple = ()
        kwargs: dict = field(default_factory=dict)
        parents: dict = field(default_factory=dict)

    class MiniTensor:
        def __init__(self, array, requires_grad=False, recipe=None):
            self.array = array; self.requires_grad = requires_grad
            self.recipe = recipe; self.grad = None

    # === cx22_topo_sort: generic DFS sanity (linear chain) ===
    class N:
        def __init__(self, name, *children):
            self.name = name; self.children = list(children)
    def get_ch(n): return n.children
    c = N('c'); b = N('b', c); a = N('a', b)
    order = cx22_topo_sort(a, get_ch)
    assert [n.name for n in order] == ['c', 'b', 'a'], 'deps-first, root LAST'

    # === cx22_topo_sort: cycle detection ===
    x = N('x'); y = N('y')
    x.children = [y]; y.children = [x]
    raised = False
    try: cx22_topo_sort(x, get_ch)
    except ValueError: raised = True
    assert raised, 'cycle must raise ValueError'

    # === cx22_sorted_graph: diamond compute graph ===
    # leaves: a, b, c; d = a*b; e = log(c); f = d*e; g = log(f).
    a = MiniTensor(t.tensor([1.0]), requires_grad=True)
    b = MiniTensor(t.tensor([2.0]), requires_grad=True)
    c = MiniTensor(t.tensor([3.0]), requires_grad=True)
    d = MiniTensor(a.array * b.array, requires_grad=True)
    d.recipe = Recipe(func=t.multiply, args=(a.array, b.array), kwargs={}, parents={0: a, 1: b})
    e = MiniTensor(t.log(c.array), requires_grad=True)
    e.recipe = Recipe(func=t.log, args=(c.array,), kwargs={}, parents={0: c})
    f = MiniTensor(d.array * e.array, requires_grad=True)
    f.recipe = Recipe(func=t.multiply, args=(d.array, e.array), kwargs={}, parents={0: d, 1: e})
    g = MiniTensor(t.log(f.array), requires_grad=True)
    g.recipe = Recipe(func=t.log, args=(f.array,), kwargs={}, parents={0: f})

    order = cx22_sorted_graph(g)

    # === end node FIRST ===
    assert order[0] is g, f'first must be end node g; got {order[0]}'

    # === all 7 unique nodes present ===
    assert len(order) == 7, f'expected 7 nodes, got {len(order)}'
    ids = {id(n) for n in order}
    assert ids == {id(a), id(b), id(c), id(d), id(e), id(f), id(g)}

    # === every node appears BEFORE its parents (reverse-topo invariant) ===
    pos = {id(n): i for i, n in enumerate(order)}
    assert pos[id(g)] < pos[id(f)], 'g before f'
    assert pos[id(f)] < pos[id(d)], 'f before d'
    assert pos[id(f)] < pos[id(e)], 'f before e'
    assert pos[id(d)] < pos[id(a)], 'd before a'
    assert pos[id(d)] < pos[id(b)], 'd before b'
    assert pos[id(e)] < pos[id(c)], 'e before c'

    # === singleton (just a leaf) ===
    lonely = MiniTensor(t.tensor([5.0]), requires_grad=True)
    assert cx22_sorted_graph(lonely) == [lonely], 'singleton'

    # === composition with cx22_topo_sort: the sorted_graph wrapper IS
    # `topo_sort(tensor, get_parents)[::-1]` — confirm by manual call
    def get_parents_fn(t_):
        if t_.recipe is None: return []
        return list(t_.recipe.parents.values())
    manual = cx22_topo_sort(g, get_parents_fn)[::-1]
    assert [id(n) for n in manual] == [id(n) for n in cx22_sorted_graph(g)], (
        'cx22_sorted_graph must be cx22_topo_sort(...)[::-1]')
    _dd_passed.add('cx22')

_test_cx22()

<details><summary>Show solution — cx22</summary>

```python
def cx22_topo_sort(root, get_children):
    result = []
    perm = set(); temp = set()
    def visit(node):
        nid = id(node)
        if nid in perm: return
        if nid in temp:
            raise ValueError(f'cycle at {node!r}')
        temp.add(nid)
        for child in get_children(node):
            visit(child)
        temp.remove(nid); perm.add(nid)
        result.append(node)
    visit(root)
    return result

def cx22_sorted_graph(tensor):
    def get_parents(t_):
        if t_.recipe is None: return []
        return list(t_.recipe.parents.values())
    return cx22_topo_sort(tensor, get_parents)[::-1]
```

**The two atoms split GENERIC sort from DOMAIN walk.** `dfs-three-set-toposort` knows nothing about MiniTensors — it just needs a `get_children` callable. `sorted-computational-graph` supplies that callable for the MiniTensor recipe graph. Same DFS engine, different traversal rule.

**`[::-1]` flips deps-first into end-first.** The generic sort appends each node AFTER visiting all its children — deps-first, root last. The reverse pass wants the OPPOSITE — start at the loss node, walk to the leaves. Reversing is O(N) and keeps both orderings derivable from one helper.

**`get_parents` on leaves returns `[]`.** A leaf has `recipe is None` — no upstream tensors to walk to. Returning `[]` is the DFS termination signal; the visit appends the leaf and the recursion unwinds.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx22'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx22',
        'subtopics': ["Backprop: DFS three-set toposort", "Backprop: Sorted computation graph"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()